# Naive feature extraction - 12 카테고리 (~241 features)

Naive 파이프라인 (01-registration alignment 사용 X):
- Zone 정의: 각 visit 의 자기 fovea + disc + disc_diameter (per-image)
- Mask: base (unaligned) vessel/av/disc

**기존 6 카테고리 (188)** + **신규 Tier 1+2 (53)** = **~241 features**

**신규**:
- Tortuosity 12 (3 net x mean/median/max/p90)
- FAZ 5 (area/diameter/circularity/perimeter)
- Crossings 9 (whole + Zone B/C x n/area/mean_cluster)
- Bifurcation angle 12 (3 net x mean/std/dev_optimal/n)
- Width variability 6 (3 net x cov_mean/cov_max)
- Junction exponent 9 (Murray X_mean/std/n)

전제: 01-registration.ipynb 의 inventory.csv 필요.
비교: 03-features-compare.ipynb 에서 aligned 와 비교.


## 0. Imports + 경로 + features 모듈


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

PKG_ROOT = Path("..").resolve()
if str(PKG_ROOT) not in sys.path:
    sys.path.insert(0, str(PKG_ROOT))

from features.zones import (
    make_etdrs_9subfields, make_disc_zones, mm_per_px_from_disc,
    ETDRS_9_NAMES, DISC_ZONE_NAMES,
    DEFAULT_ETDRS_RADII_MM, DEFAULT_DISC_MARGIN_ZONES, STANDARD_DISC_MM,
)
from features.density  import zone_density_features, whole_image_density_features
from features.skeleton import (
    whole_image_skeleton_features, whole_image_segment_features,
    whole_image_tortuosity_features, whole_image_bifurcation_features,
)
from features.caliber  import (
    whole_image_caliber_features, compute_crae_crve_avr,
    whole_image_width_variability, whole_image_junction_features,
)
from features.topology import whole_image_fractal_features, sholl_features
from features.color    import whole_image_color_features
from features.faz      import compute_faz
from features.crossings import crossings_features


In [ ]:
SQUARE_SIZE = 512
src_name = "20230118_HCS_Fundus_image"

rgb_path      = Path(f"g:/RAG/project_1/output/preprocessed_{SQUARE_SIZE}") / src_name / "rgb"
out_root      = Path(f"g:/RAG/project_1/output/vascx_output_{SQUARE_SIZE}") / src_name
av_path       = out_root / "av"
vessels_path  = out_root / "vessels"
discs_path    = out_root / "discs"
aligned_root  = Path(f"g:/RAG/project_1/output/aligned_{SQUARE_SIZE}") / src_name
features_out  = Path(f"g:/RAG/project_1/output/features_json_naive_{SQUARE_SIZE}") / src_name
features_out.mkdir(parents=True, exist_ok=True)

assert (aligned_root / "inventory.csv").exists(), "01-registration.ipynb 먼저 실행"
print("features out:", features_out)


## 1. Inventory 로드 (fovea, disc_diameter per-visit)


In [ ]:
inv_df = pd.read_csv(aligned_root / "inventory.csv", parse_dates=["date"])
print(f"inv_df: {len(inv_df)} rows")

inventory = {}
for (pid, eye), grp in inv_df.groupby(["patient", "eye"], sort=False):
    inventory.setdefault(pid, {})[eye] = grp.sort_values(["date", "shot"]).to_dict("records")

for pid, by_eye in inventory.items():
    for eye, visits in by_eye.items():
        # per-image disc_diameter 편차 확인
        dds = [v["disc_diameter"] for v in visits]
        print(f"{pid}/{eye}: {len(visits)} visits, disc_diameter={np.mean(dds):.1f}±{np.std(dds):.1f}px")


## 2. Naive zone feature 계산 — 각 visit 의 자기 reference 사용


In [ ]:
def fundus_valid(rgb): return rgb.sum(-1) > 10

group_dd = inv_df.groupby(["patient","eye"])["disc_diameter"].mean().to_dict()

feat_rows = []
for pid, by_eye in tqdm(inventory.items(), desc="patients"):
    for eye, visits in by_eye.items():
        if eye == "unknown": continue

        dd_grp = group_dd.get((pid, eye), np.nan)
        mpp_grp = mm_per_px_from_disc(dd_grp) if (np.isfinite(dd_grp) and dd_grp >= 5) else np.nan

        for v in visits:
            vid = v["id"]
            try:
                rgb  = np.array(Image.open(rgb_path      / f"{vid}.png"))
                vs_m = np.array(Image.open(vessels_path / f"{vid}.png"))
                av_m = np.array(Image.open(av_path      / f"{vid}.png"))
                dc_m = np.array(Image.open(discs_path   / f"{vid}.png"))
            except FileNotFoundError:
                continue

            fx, fy   = v["x_fovea"], v["y_fovea"]
            dcx, dcy = v["x_disc"],  v["y_disc"]

            dd_v = v.get("disc_diameter", np.nan)
            if np.isfinite(dd_v) and dd_v >= 5:
                mpp_v = mm_per_px_from_disc(dd_v)
            else:
                dd_v, mpp_v = float(dd_grp), float(mpp_grp)
            if not np.isfinite(dd_v) or dd_v < 5:
                continue

            etdrs  = make_etdrs_9subfields(fx, fy, eye, mpp_v, (SQUARE_SIZE, SQUARE_SIZE), DEFAULT_ETDRS_RADII_MM)
            disc_z = make_disc_zones(dcx, dcy, dd_v, (SQUARE_SIZE, SQUARE_SIZE), DEFAULT_DISC_MARGIN_ZONES)
            zones  = {**etdrs, **disc_z}
            valid  = fundus_valid(rgb)

            # Existing 6
            w_dens   = whole_image_density_features(vs_m, av_m, dc_m, valid_mask=valid)
            z_dens   = zone_density_features(vs_m, av_m, dc_m, zones, valid_mask=valid)
            w_skel   = whole_image_skeleton_features(vs_m, av_m, valid_mask=valid)
            w_seg    = whole_image_segment_features(vs_m, av_m, valid_mask=valid)
            w_cal    = whole_image_caliber_features(vs_m, av_m, valid_mask=valid, mm_per_px=mpp_v)
            knudtson = compute_crae_crve_avr(av_m, zones["zone_B"], mm_per_px=mpp_v)
            w_frac   = whole_image_fractal_features(vs_m, av_m, valid_mask=valid)
            w_sh_f   = sholl_features(vs_m, av_m, (fx, fy),   valid_mask=valid, center_name="fovea")
            w_sh_d   = sholl_features(vs_m, av_m, (dcx, dcy), valid_mask=valid, center_name="disc")
            w_color  = whole_image_color_features(rgb, vs_m, av_m, valid_mask=valid)

            # New Tier 1+2
            w_tort   = whole_image_tortuosity_features(vs_m, av_m, valid_mask=valid)
            faz      = compute_faz(vs_m, (fx, fy), valid_mask=valid, mm_per_px=mpp_v, disc_diameter_px=dd_v)
            cross    = crossings_features(av_m, zones=zones, valid_mask=valid)
            w_bif    = whole_image_bifurcation_features(vs_m, av_m, valid_mask=valid)
            w_wv     = whole_image_width_variability(vs_m, av_m, valid_mask=valid)
            w_junc   = whole_image_junction_features(vs_m, av_m, valid_mask=valid)

            feat_rows.append({
                "id": vid, "patient": pid, "eye": eye,
                "date": v["date"], "shot": v["shot"],
                "disc_diameter_px": dd_v, "mm_per_px": mpp_v,
                **w_dens, **z_dens, **w_skel, **w_seg, **w_cal, **knudtson,
                **w_frac, **w_sh_f, **w_sh_d, **w_color,
                **w_tort, **faz, **cross, **w_bif, **w_wv, **w_junc,
            })

features_zones_naive = pd.DataFrame(feat_rows).sort_values(["patient","eye","date"]).reset_index(drop=True)

# ── 환자별 JSON 저장 (mergeable to CSV) ──
import json as _json
from datetime import datetime as _dt

def _jd(o):
    if isinstance(o, (np.integer,)):  return int(o)
    if isinstance(o, (np.floating,)):
        v = float(o); return None if not np.isfinite(v) else v
    if isinstance(o, np.ndarray):     return o.tolist()
    if isinstance(o, pd.Timestamp):   return o.isoformat()
    return str(o)

n_saved = 0
for pid, pdf in features_zones_naive.groupby("patient", sort=False):
    eye_meta = {}
    for eye in pdf["eye"].unique():
        vg = inv_df[(inv_df["patient"] == pid) & (inv_df["eye"] == eye)]
        if len(vg) == 0: continue
        dd_mean = float(vg["disc_diameter"].mean())
        eye_meta[eye] = {
            "fovea_x": float(vg["x_fovea"].mean()), "fovea_y": float(vg["y_fovea"].mean()),
            "disc_cx": float(vg["x_disc"].mean()),  "disc_cy": float(vg["y_disc"].mean()),
            "disc_diameter_px": dd_mean,
            "mm_per_px": float(mm_per_px_from_disc(dd_mean)) if np.isfinite(dd_mean) and dd_mean >= 5 else None,
            "n_visits": int(len(vg)),
            "note": "naive mode — per-visit fovea/disc 사용, meta 는 group mean",
        }
    payload = {
        "patient": str(pid),
        "config": {"mode": "naive", "square_size": SQUARE_SIZE,
                   "extracted_at": _dt.utcnow().isoformat() + "Z"},
        "meta_by_eye": eye_meta,
        "features": pdf.assign(date=pdf["date"].astype(str)).to_dict("records"),
    }
    with open(features_out / f"{pid}.json", "w", encoding="utf-8") as f:
        _json.dump(payload, f, ensure_ascii=False, indent=2, default=_jd)
    n_saved += 1

# 편의용 merged CSV (downstream viz/compare 셀 호환)
features_zones_naive.to_csv(features_out / "_merged_features_naive.csv", index=False)
print(f"저장: {n_saved} patient JSON → {features_out}")
print(f"       + _merged_features_naive.csv (rows={len(features_zones_naive)}, cols={features_zones_naive.shape[1]})")
features_zones_naive.head()
